# 04 BM25

## Objetivos

* Implementar BM25 y comparar su ranking con el de TF-IDF + coseno sobre la misma colección de documentos.
* Calcular el aporte de cada término en BM25, identificar qué controlan $k_1$ y $b$, y justificar un cambio de posición usando las estadísticas de los documentos.

**1. Carga el corpus de Project Gutenberg**

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Recupere una tabla con los campos **Identificador de documento | Título | Texto**

Calcule la columna `tokens` para la tabla, con lista de tokens normalizados de cada documento.

Calcula DF e IDF.

In [1]:
from pathlib import Path
from collections import Counter
import re
import string
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
ruta_carpeta = Path("cache/epub")
limite_documentos = 100
puntuacion = str.maketrans("", "", string.punctuation)

def preprocesamiento_datos(texto):
    # Eliminar puntuación, convertir a minúsculas y dividir en tokens
    return texto.lower().translate(puntuacion).split()

def cargar_corpus(carpeta, limite):
    documentos = []
    for ruta in sorted(carpeta.rglob("*.txt")):
        texto = ruta.read_text(encoding="utf-8-sig", errors="replace")
        tokens = preprocesamiento_datos(texto)
        if not tokens:
            continue
        titulo = re.search(r"^Title:[ \t]*(.+)$", texto, re.MULTILINE | re.IGNORECASE) # Buscar el título en el texto
        documentos.append({
            "id": ruta.relative_to(carpeta).as_posix(),
            "titulo": titulo[1].strip() if titulo else ruta.stem,
            "texto": texto,
            "tokens": tokens,
        })
        if len(documentos) == limite:
            break
    if not documentos:
        raise ValueError(f"No se encontraron documentos con texto en {carpeta.resolve()}")
    return documentos

In [3]:
# Calcula la frecuencia documental (DF) de cada término del corpus.
def contar_df(frecuencias):
    df = Counter()
    for tf in frecuencias:
        df.update(tf.keys())
    return df

# Entrega a scikit-learn los tokens ya normalizados.
def conservar_tokens(tokens):
    return tokens

corpus = pd.DataFrame(cargar_corpus(ruta_carpeta, limite_documentos))
# Muestra hasta cinco documentos con los primeros 100 caracteres y 10 tokens de cada uno.
display(corpus.assign(
    texto=corpus["texto"].str.slice(0, 100),
    tokens=corpus["tokens"].str[:10],
).head())
print("Vista previa: corpus conserva el texto completo y todos los tokens.")

# Cuenta las apariciones de cada término por documento (TF).
frecuencias = [Counter(tokens) for tokens in corpus["tokens"]]
df = contar_df(frecuencias)
n = len(corpus)
# Longitud de cada documento en tokens.
longitudes = corpus["tokens"].map(len).to_numpy()
longitud_media = longitudes.mean()
# IDF de TF-IDF = ln(N / DF) + 1.
idf_tfidf = {t: np.log(n / cantidad) + 1 for t, cantidad in df.items()}
# IDF de BM25 = ln(1 + (N - DF + 0.5) / (DF + 0.5)).
idf_bm25 = {t: np.log1p((n - cantidad + 0.5) / (cantidad + 0.5))
            for t, cantidad in df.items()}
print(f"{n} documentos, {len(df):,} términos y longitud media de {longitud_media:,.2f} tokens.")

,id,titulo,texto,tokens
0,10/pg10.txt,The King James Version of the Bible,The Project Gutenberg eBook of The King James ...,"[the, project, gutenberg, ebook, of, the, king..."
1,100/pg100.txt,The Complete Works of William Shakespeare,The Project Gutenberg eBook of The Complete Wo...,"[the, project, gutenberg, ebook, of, the, comp..."
2,1000/pg1000.txt,La Divina Commedia di Dante: Complete,The Project Gutenberg eBook of La Divina Comme...,"[the, project, gutenberg, ebook, of, la, divin..."
3,10000/pg10000.txt,The Magna Carta,The Project Gutenberg eBook of The Magna Carta...,"[the, project, gutenberg, ebook, of, the, magn..."
4,10001/pg10001.txt,Apocolocyntosis,The Project Gutenberg eBook of Apocolocyntosis...,"[the, project, gutenberg, ebook, of, apocolocy..."


Vista previa: corpus conserva el texto completo y todos los tokens.
100 documentos, 172,234 términos y longitud media de 73,182.07 tokens.


**2. Benchmarking**

Seleccione **ocho consultas en el idioma del corpus** y registra en una línea qué busca el usuario con cada una. Incluye términos muy frecuentes, términos poco frecuentes, y consultas de distintas longitudes. Usa $df$ para justificar las elecciones.

Para **TF-IDF + coseno** conserva la representación dispersa de la matriz.

Para **BM25** define $k_1=1.2$, $b=0.75$. Mantén intactas las estadísticas de la colección al cambiar de consulta.

Para cada consulta, ordena por puntuación descendente y desempata por `doc_id` ascendente (trata los IDs como cadenas en ambos métodos). Presenta hasta diez documentos con score positivo por método. Una consulta sin términos en el vocabulario produce dos listas vacías.

In [4]:
# Calcula el aporte de cada término de la consulta a todos los documentos.
def aportes_bm25(tokens, k1=1.2, b=0.75):
    # Con b=0, el ajuste queda en k1 para todos los libros.
    ajuste = k1 * (1 - b + b * longitudes / longitud_media)
    aportes = {}
    for termino in dict.fromkeys(tokens):
        if termino in idf_bm25:
            tf = np.array([conteo[termino] for conteo in frecuencias])
            # Las repeticiones adicionales pesan cada vez menos por la saturación de TF.
            aportes[termino] = idf_bm25[termino] * tf * (k1 + 1) / (tf + ajuste)
    return aportes

# Suma los aportes por término para obtener el score BM25 de cada documento.
def sumar_aportes(aportes):
    return sum(aportes.values(), np.zeros(n))

def ordenar_resultados(scores):
    # Se excluyen los scores cero. En empate, se ordena por ID ascendente.
    positivos = np.flatnonzero(scores > 0)
    return sorted(positivos, key=lambda i: (-scores[i], corpus.at[i, "id"]))


In [5]:

consultas = {
    "the": "Medir qué devuelve una palabra funcional muy común y poco específica.",
    "whale": "Encontrar libros donde se mencionen ballenas.",
    "king queen": "Buscar relatos con reyes y reinas.",
    "faith prayer god": "Buscar textos sobre fe y oración.",
    "war army battle": "Encontrar relatos de guerras y batallas.",
    "whale captain ship sea": "Buscar relatos marítimos con capitanes y ballenas.",
    "gondokoro nile": "Encontrar exploraciones del Nilo que mencionen Gondokoro.",
    "zzzxxyyfictional": "Comprobar cómo se responde a un término inexistente.",
}
tokens_consultas = {q: preprocesamiento_datos(q) for q in consultas}

# Clasifica cada término según la proporción de documentos donde aparece.
def describir_df(termino):
    cantidad = df[termino]
    if cantidad == 0:
        categoria = "fuera del vocabulario"
    elif cantidad <= 0.10 * n:
        categoria = "poco frecuente"
    elif cantidad >= 0.80 * n:
        categoria = "muy frecuente"
    else:
        categoria = "frecuencia intermedia"
    return f"{termino}: {cantidad}/{n} ({categoria})"

display(pd.DataFrame([
    {"Consulta": q, "Intención": intencion, "Longitud": len(tokens_consultas[q]),
     "DF por término": " | ".join(map(describir_df, tokens_consultas[q]))}
    for q, intencion in consultas.items()
]))

vectorizador = TfidfVectorizer(analyzer=conservar_tokens, smooth_idf=False, norm="l2")
matriz = vectorizador.fit_transform(corpus["tokens"])
# La consulta usa el vocabulario y el IDF aprendidos de los documentos.
matriz_consultas = vectorizador.transform(list(tokens_consultas.values()))
assert np.allclose(vectorizador.idf_, [idf_tfidf[t] for t in vectorizador.get_feature_names_out()])
cosenos = cosine_similarity(matriz_consultas, matriz, dense_output=False)

resultados = {}
for i, (consulta, tokens) in enumerate(tokens_consultas.items()):
    scores_coseno = cosenos[i].toarray().ravel()
    aportes = aportes_bm25(tokens)
    scores_bm25 = sumar_aportes(aportes)
    resultados[consulta] = {
        "coseno": scores_coseno, "bm25": scores_bm25, "aportes": aportes,
        "orden_tfidf": ordenar_resultados(scores_coseno),
        "orden_bm25": ordenar_resultados(scores_bm25),
    }

,Consulta,Intención,Longitud,DF por término
0,the,Medir qué devuelve una palabra funcional muy c...,1,the: 100/100 (muy frecuente)
1,whale,Encontrar libros donde se mencionen ballenas.,1,whale: 13/100 (frecuencia intermedia)
2,king queen,Buscar relatos con reyes y reinas.,2,king: 71/100 (frecuencia intermedia) | queen: ...
3,faith prayer god,Buscar textos sobre fe y oración.,3,faith: 67/100 (frecuencia intermedia) | prayer...
4,war army battle,Encontrar relatos de guerras y batallas.,3,war: 76/100 (frecuencia intermedia) | army: 58...
5,whale captain ship sea,Buscar relatos marítimos con capitanes y balle...,4,whale: 13/100 (frecuencia intermedia) | captai...
6,gondokoro nile,Encontrar exploraciones del Nilo que mencionen...,2,gondokoro: 1/100 (poco frecuente) | nile: 19/1...
7,zzzxxyyfictional,Comprobar cómo se responde a un término inexis...,1,zzzxxyyfictional: 0/100 (fuera del vocabulario)


**3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |

Calcula cuántos IDs comparten los dos top 10 con $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$. Reporta también el tamaño real de cada lista y el primer documento de cada método. Dos listas pueden tener muchos IDs en común y ordenarlos de forma distinta.

Elige una consulta y dos documentos que cambien de posición o cuya distancia en el ranking cambie. Registra $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de consulta para ambos documentos. Relaciona esas cifras con la saturación, la longitud o la rareza del término. Usa los scores de cada método para explicar su propio ranking.

In [6]:
def etiqueta(i):
    return f"{corpus.at[i, 'id']} · {corpus.at[i, 'titulo']}"

def posiciones(orden):
    return {i: posicion for posicion, i in enumerate(orden, start=1)}

# Alinea ambos top 10 por posición y cuenta los documentos compartidos.
def comparar_top(resultado):
    top_tfidf = resultado["orden_tfidf"][:10]
    top_bm25 = resultado["orden_bm25"][:10]
    tabla = pd.DataFrame({
        "Posición": range(1, max(len(top_tfidf), len(top_bm25)) + 1),
        "ID y título TF-IDF": pd.Series([etiqueta(i) for i in top_tfidf], dtype=str),
        "Coseno": pd.Series([resultado["coseno"][i] for i in top_tfidf], dtype=float),
        "ID y título BM25": pd.Series([etiqueta(i) for i in top_bm25], dtype=str),
        "Score BM25": pd.Series([resultado["bm25"][i] for i in top_bm25], dtype=float),
    })
    # La intersección cuenta libros compartidos aunque aparezcan en distintos puestos.
    ids_tfidf = {corpus.at[i, "id"] for i in top_tfidf}
    ids_bm25 = {corpus.at[i, "id"] for i in top_bm25}
    return tabla, {
        "Tamaño TF-IDF": len(ids_tfidf), "Tamaño BM25": len(ids_bm25),
        "IDs en común": len(ids_tfidf & ids_bm25),
        "Primero TF-IDF": etiqueta(top_tfidf[0]) if top_tfidf else "Sin resultados",
        "Primero BM25": etiqueta(top_bm25[0]) if top_bm25 else "Sin resultados",
    }

In [7]:
# Muestra TF, DF, IDF y aporte BM25 por término y documento.
def tabla_aportes(tokens, indices, aportes):
    return pd.DataFrame([
        {"Documento": etiqueta(i), "Longitud": longitudes[i], "Término": termino,
         "TF": frecuencias[i][termino], "DF": df[termino],
         "IDF BM25": idf_bm25.get(termino, 0.0),
         "Aporte BM25": aportes[termino][i] if termino in aportes else 0.0}
        for i in indices for termino in dict.fromkeys(tokens)
    ])

In [8]:
resumen = []
for consulta, resultado in resultados.items():
    print(consulta)
    tabla, comparacion = comparar_top(resultado)
    display(tabla)
    resumen.append({"Consulta": consulta, **comparacion})
display(pd.DataFrame(resumen))


# Selecciona dos documentos cuyo orden relativo cambia entre los rankings.
consulta_caso = "war army battle"
indices_por_titulo = {titulo: i for i, titulo in corpus["titulo"].items()}
a = indices_por_titulo["The Atlantic Monthly, Volume 10, No. 60, October, 1862"]
b = indices_por_titulo["The Iron Game: A Tale of the War"]
caso = resultados[consulta_caso]
tokens_caso = tokens_consultas[consulta_caso]
pos_tfidf = posiciones(caso["orden_tfidf"])
pos_bm25 = posiciones(caso["orden_bm25"])

print(f"Consulta analizada: {consulta_caso}")
display(pd.DataFrame([
    {"Documento": etiqueta(i), "Posición TF-IDF": pos_tfidf[i],
     "Coseno": caso["coseno"][i], "Posición BM25": pos_bm25[i],
     "Score BM25": caso["bm25"][i]}
    for i in (a, b)
]))
# El desglose muestra qué términos explican la diferencia de score BM25.
display(tabla_aportes(tokens_caso, (a, b), caso["aportes"]))


the


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,10071/pg10071.txt · By-Ways of Bombay,0.745177,10/pg10.txt · The King James Version of the Bible,0.010916
1,2,10058/pg10058.txt · The Divine Office: A Study...,0.735368,10058/pg10058.txt · The Divine Office: A Study...,0.010916
2,3,10038/pg10038.txt · The Magnetic North,0.721656,10060/pg10060.txt · Discourses: Biological & G...,0.010916
3,4,10012/pg10012.txt · The Mountains of California,0.720973,10012/pg10012.txt · The Mountains of California,0.010916
4,5,10065/pg10065.txt · The Constitution of the Un...,0.718818,10038/pg10038.txt · The Magnetic North,0.010916
5,6,10044/pg10044.txt · The Father of British Cana...,0.710940,10062/pg10062.txt · The Iron Game: A Tale of t...,0.010916
6,7,10062/pg10062.txt · The Iron Game: A Tale of t...,0.710356,"10088/pg10088.txt · Thaumaturgia; Or, Elucidat...",0.010916
7,8,10043/pg10043.txt · Klondyke Nuggets: A Brief ...,0.703799,"10009/pg10009.txt · Wild Northern Scenes; Or, ...",0.010916
8,9,10080/pg10080.txt · Mobilizing Woman-Power,0.703305,10085/pg10085.txt · Moorish Literature,0.010916
9,10,10060/pg10060.txt · Discourses: Biological & G...,0.703054,10084/pg10084.txt · Kazan,0.010916


whale


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,"10015/pg10015.txt · Punchinello, Volume 1, No....",0.024018,"10015/pg10015.txt · Punchinello, Volume 1, No....",4.249095
1,2,10040/pg10040.txt · Aleutian Indian and Englis...,0.006910,10040/pg10040.txt · Aleutian Indian and Englis...,3.268847
2,3,"10036/pg10036.txt · Punchinello, Volume 2, No....",0.001944,"10009/pg10009.txt · Wild Northern Scenes; Or, ...",3.014087
3,4,"10018/pg10018.txt · Punchinello, Volume 1, No....",0.001887,"10018/pg10018.txt · Punchinello, Volume 1, No....",2.941645
4,5,"10009/pg10009.txt · Wild Northern Scenes; Or, ...",0.000905,"10036/pg10036.txt · Punchinello, Volume 2, No....",2.937253
5,6,"1005/pg1005.txt · Divine Comedy, Cary's Transl...",0.000899,10041/pg10041.txt · The Rivet in Grandfather's...,2.746364
6,7,10041/pg10041.txt · The Rivet in Grandfather's...,0.000792,"10079/pg10079.txt · The Atlantic Monthly, Volu...",2.619565
7,8,"10079/pg10079.txt · The Atlantic Monthly, Volu...",0.000662,"1005/pg1005.txt · Divine Comedy, Cary's Transl...",2.477354
8,9,10020/pg10020.txt · The Strand Magazine: Vol. ...,0.000652,10020/pg10020.txt · The Strand Magazine: Vol. ...,2.342922
9,10,100/pg100.txt · The Complete Works of William ...,0.000311,10085/pg10085.txt · Moorish Literature,1.797790


king queen


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,100/pg100.txt · The Complete Works of William ...,0.049397,100/pg100.txt · The Complete Works of William ...,2.058771
1,2,10081/pg10081.txt · The Boy Allies at Jutland;...,0.022560,10085/pg10085.txt · Moorish Literature,2.004304
2,3,10085/pg10085.txt · Moorish Literature,0.021602,10003/pg10003.txt · My First Years as a French...,2.002319
3,4,10/pg10.txt · The King James Version of the Bible,0.019406,10022/pg10022.txt · White Queen of the Canniba...,1.961719
4,5,10056/pg10056.txt · Chinese literature,0.019293,"10079/pg10079.txt · The Atlantic Monthly, Volu...",1.944453
5,6,10000/pg10000.txt · The Magna Carta,0.010652,10024/pg10024.txt · Beneath the Banner: Being ...,1.931794
6,7,10022/pg10022.txt · White Queen of the Canniba...,0.009739,10020/pg10020.txt · The Strand Magazine: Vol. ...,1.929422
7,8,10003/pg10003.txt · My First Years as a French...,0.009392,10056/pg10056.txt · Chinese literature,1.893842
8,9,"10039/pg10039.txt · The Works of Aphra Behn, V...",0.006884,"10088/pg10088.txt · Thaumaturgia; Or, Elucidat...",1.869866
9,10,10020/pg10020.txt · The Strand Magazine: Vol. ...,0.006018,10000/pg10000.txt · The Magna Carta,1.865872


... (10 outputs left out)

,Consulta,Tamaño TF-IDF,Tamaño BM25,IDs en común,Primero TF-IDF,Primero BM25
0,the,10,10,5,10071/pg10071.txt · By-Ways of Bombay,10/pg10.txt · The King James Version of the Bible
1,whale,10,10,9,"10015/pg10015.txt · Punchinello, Volume 1, No....","10015/pg10015.txt · Punchinello, Volume 1, No...."
2,king queen,10,10,7,100/pg100.txt · The Complete Works of William ...,100/pg100.txt · The Complete Works of William ...
3,faith prayer god,10,10,8,10058/pg10058.txt · The Divine Office: A Study...,10058/pg10058.txt · The Divine Office: A Study...
4,war army battle,10,10,7,"10077/pg10077.txt · The Atlantic Monthly, Volu...",10062/pg10062.txt · The Iron Game: A Tale of t...
5,whale captain ship sea,10,10,2,10008/pg10008.txt · The Mystery,"10015/pg10015.txt · Punchinello, Volume 1, No...."
6,gondokoro nile,10,10,4,10024/pg10024.txt · Beneath the Banner: Being ...,10024/pg10024.txt · Beneath the Banner: Being ...
7,zzzxxyyfictional,0,0,0,Sin resultados,Sin resultados


Consulta analizada: war army battle


,Documento,Posición TF-IDF,Coseno,Posición BM25,Score BM25
0,"10077/pg10077.txt · The Atlantic Monthly, Volu...",1,0.033578,2,2.880283
1,10062/pg10062.txt · The Iron Game: A Tale of t...,4,0.018449,1,2.881360


,Documento,Longitud,Término,TF,DF,IDF BM25,Aporte BM25
0,"10077/pg10077.txt · The Atlantic Monthly, Volu...",87114,war,122,76,0.277830,0.604431
1,"10077/pg10077.txt · The Atlantic Monthly, Volu...",87114,army,204,58,0.546094,1.193384
2,"10077/pg10077.txt · The Atlantic Monthly, Volu...",87114,battle,33,60,0.512477,1.082467
3,10062/pg10062.txt · The Iron Game: A Tale of t...,144583,war,167,76,0.277830,0.603713
4,10062/pg10062.txt · The Iron Game: A Tale of t...,144583,army,97,58,0.546094,1.176208
5,10062/pg10062.txt · The Iron Game: A Tale of t...,144583,battle,88,60,0.512477,1.101440


Para la consulta war army battle, TF-IDF + coseno ubicó a The Atlantic Monthly en el primer puesto con 0,033578 y a The Iron Game en el cuarto con 0,018449. El primer documento tiene 204 apariciones de army, frente a 97 en el segundo. Ese conteo aumenta el peso de army en su vector TF-IDF. El coseno divide el producto punto por la norma de ambos vectores, así que el puesto depende de cuánto pesan los términos de la consulta en relación con el vector completo del documento.

En BM25, el IDF indica qué tan distintivo es un término en la colección. war aparece en 76 de los 100 documentos, por lo que tiene el menor IDF (0,277830). Aunque aparece 167 veces en The Iron Game y 122 en The Atlantic Monthly, sus aportes son casi iguales (0,603713 y 0,604431). Esto se explica por tres factores: el IDF bajo limita el peso de war, la saturación reduce el beneficio de repetirlo y el ajuste por longitud exige más apariciones para lograr el mismo aporte en The Iron Game, que tiene 144 583 tokens frente a 87 114.

army aparece en menos documentos (58) y tiene un IDF mayor (0,546094). Por eso es el término que más aporta en ambos textos. Sin embargo, pasar de 97 a 204 apariciones solo aumenta el aporte de 1,176208 a 1,193384. El IDF es el mismo para ambos documentos. Lo que explica ese aumento pequeño es que, con frecuencias tan altas, ambos aportes ya están cerca del límite de saturación.

El término que decide el orden es battle. Aparece en 60 documentos y tiene un IDF de 0,512477, cercano al de army. Sus 88 apariciones en The Iron Game le dan una ventaja de 0,018973 en ese término. Esa ventaja supera los 0,017894 que The Atlantic Monthly gana en conjunto con war y army, usando los aportes redondeados de la tabla. Por ese margen, The Iron Game pasa al primer puesto con 2,881360, seguido de The Atlantic Monthly con 2,880283.

**4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.

In [9]:
configuraciones = [
    ("Base", 1.2, 0.75),
    ("Sin ajuste por longitud", 1.2, 0.0),
    ("Mayor efecto de las repeticiones", 2.0, 0.75),
]

# Compara las posiciones y los aportes de los mismos documentos al variar k1 y b.
def comparar_parametros(tokens, indices, configuraciones):
    filas = []
    detalles = []
    # En cada prueba se reutilizan el DF, el IDF y las longitudes del corpus.
    for nombre, k1, ajuste_longitud in configuraciones:
        aportes = aportes_bm25(tokens, k1=k1, b=ajuste_longitud)
        scores = sumar_aportes(aportes)
        orden = posiciones(ordenar_resultados(scores))
        for i in indices:
            filas.append({
                "Configuración": nombre, "k1": k1, "b": ajuste_longitud,
                "Documento": etiqueta(i), "Longitud": longitudes[i],
                "Posición": orden.get(i), "Score": scores[i],
            })
        # Los aportes por término pueden variar aun cuando el puesto se mantenga.
        detalles.append(tabla_aportes(tokens, indices, aportes).assign(Configuración=nombre))
    return pd.DataFrame(filas), pd.concat(detalles, ignore_index=True)

print(f"Consulta: {consulta_caso} | Longitud media: {longitud_media:,.2f} tokens")
tabla_parametros, detalle_parametros = comparar_parametros(tokens_caso, (a, b), configuraciones)
display(tabla_parametros)
display(detalle_parametros[["Configuración", "Documento", "Término", "TF", "Aporte BM25"]])



Consulta: war army battle | Longitud media: 73,182.07 tokens


,Configuración,k1,b,Documento,Longitud,Posición,Score
0,Base,1.2,0.75,"10077/pg10077.txt · The Atlantic Monthly, Volu...",87114,2,2.880283
1,Base,1.2,0.75,10062/pg10062.txt · The Iron Game: A Tale of t...,144583,1,2.881360
2,Sin ajuste por longitud,1.2,0.00,"10077/pg10077.txt · The Atlantic Monthly, Volu...",87114,4,2.887543
3,Sin ajuste por longitud,1.2,0.00,10062/pg10062.txt · The Iron Game: A Tale of t...,144583,3,2.905872
4,Mayor efecto de las repeticiones,2.0,0.75,"10077/pg10077.txt · The Atlantic Monthly, Volu...",87114,2,3.876139
5,Mayor efecto de las repeticiones,2.0,0.75,10062/pg10062.txt · The Iron Game: A Tale of t...,144583,1,3.877569


,Configuración,Documento,Término,TF,Aporte BM25
0,Base,"10077/pg10077.txt · The Atlantic Monthly, Volu...",war,122,0.604431
1,Base,"10077/pg10077.txt · The Atlantic Monthly, Volu...",army,204,1.193384
2,Base,"10077/pg10077.txt · The Atlantic Monthly, Volu...",battle,33,1.082467
3,Base,10062/pg10062.txt · The Iron Game: A Tale of t...,war,167,0.603713
4,Base,10062/pg10062.txt · The Iron Game: A Tale of t...,army,97,1.176208
5,Base,10062/pg10062.txt · The Iron Game: A Tale of t...,battle,88,1.101440
6,Sin ajuste por longitud,"10077/pg10077.txt · The Atlantic Monthly, Volu...",war,122,0.605272
7,Sin ajuste por longitud,"10077/pg10077.txt · The Atlantic Monthly, Volu...",army,204,1.194381
8,Sin ajuste por longitud,"10077/pg10077.txt · The Atlantic Monthly, Volu...",battle,33,1.087890
9,Sin ajuste por longitud,10062/pg10062.txt · The Iron Game: A Tale of t...,war,167,0.606865


## Explicación de los resultados

**BM25** Con $b=0.75$, una misma frecuencia aporta menos en un documento largo que en uno corto. Al fijar $b=0$, desaparece ese ajuste. Un $k_1$ mayor retrasa la saturación, de modo que varias apariciones pueden aumentar más el aporte. El IDF da mayor peso a términos que aparecen en pocos documentos. Las posiciones dependen de cuánto cambian los scores de todos los documentos que coinciden con la consulta.

Para `war army battle`, *The Atlantic Monthly* queda primero con TF-IDF (coseno 0,033578) y *The Iron Game* queda cuarto (0,018449). El vector del primer texto está más alineado con la consulta. Con BM25, *The Iron Game* pasa al primer puesto con 2,881360 y *The Atlantic Monthly* queda segundo con 2,880283. La diferencia es pequeña y se entiende al revisar los aportes por término.

En *The Atlantic Monthly*, `army` aparece 204 veces frente a 97 en *The Iron Game*. A pesar de esa diferencia, su aporte apenas sube de 1,17621 a 1,19338 porque BM25 reduce el peso de las repeticiones adicionales. *The Iron Game* compensa con `battle`: sus 88 apariciones aportan 1,10144, frente a 1,08247 por 33 apariciones en *The Atlantic Monthly*. Además, `war` aparece en 76 de los 100 documentos, frente a 60 para `battle` y 58 para `army`, por lo que recibe el menor IDF de los tres.

**Con `b=0`**, ambos textos ganan puntuación porque superan la media de 73.182 tokens y dejan de recibir el ajuste por longitud. *The Iron Game* tiene 144.583 tokens y sube de 2,881360 a 2,905872. *The Atlantic Monthly* tiene 87.114 tokens y sube de 2,880283 a 2,887543. Aun así, pasan de los puestos 1 y 2 a los puestos 3 y 4, respectivamente, porque otros documentos los superan al recalcular todo el ranking.

**Al subir `k1` a 2**, la saturación es más gradual. Los scores suben a 3,877569 y 3,876139, respectivamente. La ventaja de *The Iron Game* crece de aproximadamente 0,00108 a 0,00143 y ambos conservan sus puestos. En *The Iron Game*, el aporte de `battle` sube de 1,10144 a 1,47921. En *The Atlantic Monthly*, sube de 1,08247 a 1,43785. La diferencia a favor de *The Iron Game* para ese término aumenta, aunque los aportes de `army` y `war` compensan parte de esa ventaja.


## Acuerdo de uso de inteligencia artificial

Declaro que recurrí a la inteligencia artificial como apoyo para evaluar la estructura del notebook,aumentar la velocidad de escritura, reconocer posibles inconsistencias entre las indicaciones y el código. Mejorar la redacción del análisis escrito. No obstante, esta herramienta no reemplaza mi criterio ni mi responsabilidad académica. Es mi deber revisar las sugerencias, ejecutar el notebook, verificar los resultados obtenidos, comprender los algoritmos implementados y referenciar las fuentes externas consultadas. Asumo la autoría y responsabilidad del trabajo final, evitando presentar como comprobados resultados que no haya validado personalmente.